# h5_to_lerobot

이미지 데이터셋 `.h5`(`h5_add_images` 출력)를 [GR00T-N1.7-3B](https://huggingface.co/nvidia/GR00T-N1.7-3B) 파인튜닝용 **GR00T-flavored LeRobot v2.1** 데이터셋으로 포장한다. 기록된 tcp_pose를 **절대 엔드이펙터(xyz + rot6d, 9d)** 로 변환(`scripts/ee_convert.py`)해 gripper와 합친 **10d 액션**으로 쓰고, 프레임은 **mp4**, 수치는 **parquet**, 스키마·정규화·언어·`modality.json`은 **meta** 로 떨군다. 저장은 **절대 pose**이며 상대화는 GR00T가 내부에서 처리한다(`rep=RELATIVE`).

순수 **오프라인** — sim·WSL 없음. `pyarrow + imageio` 만 사용(이미 설치됨), `lerobot` 라이브러리 의존 없음 → 작동 중인 conda env 를 안 건드린다.

**`run`** — 변환 (출력 디렉토리 반환)

| 파라미터 | 설명 | 기본값 |
|---|---|---|
| `traj_path` | 데이터셋 .h5 경로 (rgb + `obs/extra/tcp_pose` 필요) | (필수) |
| `out` | 출력 디렉토리 | `None` (입력 옆 `lerobot/`) |
| `fps` | 영상 fps | `20` |
| `instruction` | `label_metadata` 키로 만드는 지시문 템플릿 (예: `"pick up the {target_id} cube"`) | `None` (디코드된 라벨 값) |
| `camera` | 사용할 카메라 | `None` (첫 번째) |

출력 구조:
```
<out>/
├── meta/{info.json, episodes.jsonl, tasks.jsonl, modality.json, stats.json}
├── data/chunk-000/episode_{i:06d}.parquet      # observation.state, action(10d 절대 EEF), 인덱스
└── videos/chunk-000/observation.images.<cam>/episode_{i:06d}.mp4
```

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from scripts.h5_to_lerobot import run

# 입력은 h5_add_images 출력(이미지 데이터셋). 먼저 /ee_verify 게이트 통과를 권장.
ds = ROOT / 'data' / 'datasets' / 'ThreeColoredCubes-v1' / 'motionplanning.rgb.pd_joint_pos.physx_cpu.h5'

# 지시문은 env 가 선언한 label_metadata 키로 생성 -> task-무관 (스크립트에 task 이름 없음).
out = run(ds, instruction='pick up the {target_id} cube', fps=20)
print('lerobot ->', out)

In [ ]:
# 산출물 점검: meta 요약 + 한 에피소드 parquet 를 DataFrame 으로
import json
import pandas as pd

info = json.loads((out / 'meta' / 'info.json').read_text(encoding='utf-8'))
print('episodes:', info['total_episodes'], '| frames:', info['total_frames'], '| tasks:', info['total_tasks'])
print('action:', info['features']['action']['shape'], info['features']['action']['names'])
print('state :', info['features']['observation.state']['shape'])
print('tasks :', (out / 'meta' / 'tasks.jsonl').read_text(encoding='utf-8'))

pq0 = sorted((out / 'data' / 'chunk-000').glob('*.parquet'))[0]
df = pd.read_parquet(pq0)
print('\n', pq0.name, '| rows =', len(df))
df.head()

CLI: `python scripts/h5_to_lerobot.py --traj-path <h5> [--instruction "pick up the {target_id} cube"] [--out DIR] [--fps 20]`

메모:
- 액션 = **10d 절대 EEF** `[eef_x,y,z, rot6d_0..5, gripper]` (`ActionFormat.XYZ_ROT6D`), 상태 = **`[qpos(9), eef_abs(9)]`** (eef 블록이 상대화 기준, `state_key="eef"`). 저장은 절대 pose(`action[t]`=다음 스텝, `state[t]`=현재) — GR00T가 `rep=RELATIVE` 로 내부 상대화. rot6d는 회전행렬 첫 두 행(GR00T `pose.py` 컨벤션). 프레임 수 N = T-1 (parquet 행수 == mp4 프레임수).
- **먼저 `/ee_verify` 게이트 통과 권장** — 검증은 7d 델타 표현으로 IK 재현을 보지만, 저장은 절대 EEF (같은 궤적의 두 표현이라 게이트 통과 = 저장 포맷 신뢰).
- `codebase_version v2.1` 타깃 (meta 스키마는 Isaac-GR00T 공식 예제 `cube_to_bowl_5`와 정합). 로더가 `stats.json` 존재를 요구 — 학습 전 GR00T env에서 `scripts/repair_lerobot_metadata.py <out> --embodiment-tag <tag>` 로 `stats.json`/`relative_stats.json` 재생성 권장. **최종 "GR00T 가 로드하나"는 학습 박스(클라우드)에서 확인.**

보통 흐름: `h5_add_images` → `h5_report`(품질) → `ee_verify`(EE 게이트) → **`h5_to_lerobot`(포장)** → GR00T 학습(클라우드). LeRobot 데이터셋 구조와 학습 한 row 의 의미는 `gr00t_n17_training_explained.ipynb` 에서 자세히 설명.